In [ ]:
import sys
import os
# Add src directory to path to import kg_voyager module
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))  # Add project root for ABSTAT package

from llm import process_subgraphs_yaml
# load dotenv
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
import yaml
from llm import process_single_subgraph_pipeline, load_prompt_template

with open("subgraphs_output/04-06-22-dbpedia-qlever.yaml") as f:
    subgraphs = yaml.safe_load(f)

# Subgraph 195: automobile / Formula One racer / company / country / Grand Prix
sg = subgraphs[195]
template = load_prompt_template()

result = process_single_subgraph_pipeline(
    subgraph=sg,
    subgraph_index=195,
    endpoint="dbpedia-qlever",
    model="gemini/gemini-3.1-pro-preview",
    template=template,
    timeout=70,
    fail_empty=False,
    verify_iterations=2,
)

print(f"Is meaningful: {result['is_meaningful']}")
if result.get("verification_status"):
    print(f"Verification: {result['verification_status']} (iteration {result.get('verify_iterations_used')})")
if result.get("most_meaningful_pair_index") is not None:
    print(f"Most meaningful pair index: {result['most_meaningful_pair_index']}")

pairs = result.get("qa_pairs", [])
for i, pair in enumerate(pairs, 1):
    print()
    print("="*60)
    print(f"Pair {i}")
    print("="*60)
    print("Original Q:", pair.get("original_question", pair.get("question", "")))
    print("Original SPARQL:")
    print(pair.get("original_sparql_query", pair.get("sparql_query", "")))
    comp = pair.get("complexity_types")
    if comp:
        print(f"[+Complexity: {comp}]")
        print("Post-Complexity Q:", pair.get("pre_verify_question", pair.get("question", "")))
        print("Post-Complexity SPARQL:")
        print(pair.get("pre_verify_sparql_query", pair.get("sparql_query", "")))
    vs = pair.get("verification_status")
    if vs:
        print(f"[Verify: {vs}]")
    rw_before = pair.get("pre_reword_question")
    rw_after = pair.get("question")
    rw_reason = pair.get("reword_reason")
    if rw_before and rw_before != rw_after:
        print("[Reword]")
        print("Before:", rw_before)
        print("After: ", rw_after)
        if rw_reason:
            print("Reason:", rw_reason)
    print("Final Q:", pair.get("question", ""))
    print("Final SPARQL:")
    print(pair.get("sparql_query", ""))
    exec_res = pair.get("execution_result", {})
    print(f"Exec: success={exec_res.get('success')} | count={exec_res.get('result_count')}")


Is meaningful: True
Verification: needs_fix (iteration None)

Pair 1
Original Q: Who was the first driver of the 1980 British Grand Prix, and what was recorded as the first driver country for that event?
Original SPARQL:
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>

SELECT ?driver ?country
WHERE {
  dbr:1980_British_Grand_Prix a dbo:GrandPrix ;
                              dbo:firstDriver ?driver ;
                              dbo:firstDriverCountry ?country .
  ?driver a dbo:FormulaOneRacer .
  ?country a dbo:Country .
}
[Verify: needs_fix]
Final Q: Which Grand Prix events have a recorded first driver and first driver country, and what are their names?
Final SPARQL:
PREFIX dbo: <http://dbpedia.org/ontology/>

SELECT ?gp ?driver ?country
WHERE {
  ?gp a dbo:GrandPrix ;
      dbo:firstDriver ?driver ;
      dbo:firstDriverCountry ?country .
}
LIMIT 10
Exec: success=None | count=None

Pair 2
Original Q: List the companies that manufacture automo

In [2]:
# Process YAML: generate questions, execute queries, save to qa_output/
output_path = process_subgraphs_yaml(
    input_yaml_path="subgraphs_output/04-06-14-dblp.yaml",
    endpoint="dblp",
    model="gemini/gemini-3.1-pro-preview",
    timeout=140,
    verify_iterations=2, 
    max_workers=20,
    max_subgraphs=150,
    focus_entity_qa=True,
    ents_ratio=0.25,
    introduce_complexity_step=True
)
print(f"Output saved to: {output_path}")

Processing 52 subgraphs [Gen + Exec + Complexity + Verify(×2) + Reword] with 20 workers...


Pipeline Progress (Gen + Exec): 100%|██████████| 52/52 [05:28<00:00,  6.31s/it]



── Verification Summary ──
  After cycle 1: 136/156 (87.2%) pairs correct
  After cycle 2: 152/156 (97.4%) pairs correct
──────────────────────────
Output saved to: subgraphs_output/qa_output/04-06-14-dblp-gemini31propreview-1.yaml


In [4]:
# Convert YAML output to JSONL filtering only successful queries with results
from yaml_to_jsonl import convert_yaml_to_jsonl

# If optimal=True, keep only the most meaningful QA per subgraph, then sample
convert_yaml_to_jsonl(output_path, num_samples=100, optimal=False, endpoint="dblp", allow_empty_results=True)

Reading subgraphs_output/qa_output/04-06-14-dblp-gemini31propreview-1.yaml...
Filtering entries...


Processing subgraphs: 100%|██████████| 52/52 [00:00<00:00, 198456.60it/s]

Sampling 100 entries from 152 candidates (49 subgraphs)...
Writing 100 entries to subgraphs_output/qa_output/04-06-14-dblp-gemini31propreview-1_sample100.jsonl...
Conversion complete.
Total QA pairs scanned: 156
Valid candidates found: 152
Entries written: 100
Output saved to: subgraphs_output/qa_output/04-06-14-dblp-gemini31propreview-1_sample100.jsonl
